In [1]:
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import torch

In [2]:
#read csv: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
veri = pd.read_csv("IMDB Dataset.csv")


In [3]:
veri.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
print(veri.columns)
print(veri.shape)
print(veri["sentiment"].value_counts())


Index(['review', 'sentiment', 'review_clean'], dtype='str')
(50000, 3)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64
Index(['review', 'sentiment', 'review_clean'], dtype='str')


In [36]:
veri["label"] = veri["sentiment"].map({
    "positive": 1,
    "negative": 0
})

metinler = veri["review"].astype(str)
y = veri["label"].values

print(len(metinler))

50000


In [37]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Kullanılan cihaz:", device)

model = model.to(device)
model.eval()

Kullanılan cihaz: cpu


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(119547, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p

In [38]:
import os
import numpy as np
import torch

os.makedirs("embedding_chunks", exist_ok=True)

batch_size = 16
chunk_size = 1000

chunk_embeddings = []
chunk_no = 0

for i in range(0, len(metinler), batch_size):

    batch_texts = metinler.iloc[i:i+batch_size].tolist()

    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    token_embeddings = outputs.last_hidden_state

    # Padding tokenlarını ortalamaya katma
    mask = inputs["attention_mask"].unsqueeze(-1)

    toplam = (token_embeddings * mask).sum(dim=1)
    adet = mask.sum(dim=1)

    batch_embeddings = toplam / adet

    chunk_embeddings.append(
        batch_embeddings.cpu().numpy()
    )

    tamamlanan = min(i + batch_size, len(metinler))

    if tamamlanan % 1000 < batch_size or tamamlanan == len(metinler):

        X_chunk = np.vstack(chunk_embeddings)

        np.save(
            f"embedding_chunks/embedding_{chunk_no:03d}.npy",
            X_chunk
        )

        print(
            tamamlanan,
            "/",
            len(metinler),
            "yorum tamamlandı ve kaydedildi."
        )

        chunk_embeddings = []
        chunk_no += 1

1008 / 50000 yorum tamamlandı ve kaydedildi.
2000 / 50000 yorum tamamlandı ve kaydedildi.
3008 / 50000 yorum tamamlandı ve kaydedildi.
4000 / 50000 yorum tamamlandı ve kaydedildi.
5008 / 50000 yorum tamamlandı ve kaydedildi.
6000 / 50000 yorum tamamlandı ve kaydedildi.
7008 / 50000 yorum tamamlandı ve kaydedildi.
8000 / 50000 yorum tamamlandı ve kaydedildi.
9008 / 50000 yorum tamamlandı ve kaydedildi.
10000 / 50000 yorum tamamlandı ve kaydedildi.
11008 / 50000 yorum tamamlandı ve kaydedildi.
12000 / 50000 yorum tamamlandı ve kaydedildi.
13008 / 50000 yorum tamamlandı ve kaydedildi.
14000 / 50000 yorum tamamlandı ve kaydedildi.
15008 / 50000 yorum tamamlandı ve kaydedildi.
16000 / 50000 yorum tamamlandı ve kaydedildi.
17008 / 50000 yorum tamamlandı ve kaydedildi.
18000 / 50000 yorum tamamlandı ve kaydedildi.
19008 / 50000 yorum tamamlandı ve kaydedildi.
20000 / 50000 yorum tamamlandı ve kaydedildi.
21008 / 50000 yorum tamamlandı ve kaydedildi.
22000 / 50000 yorum tamamlandı ve kaydedild

In [39]:
import glob
import numpy as np

dosyalar = sorted(glob.glob("embedding_chunks/embedding_*.npy"))

X_list = []

for dosya in dosyalar:
    X_list.append(np.load(dosya))

X_all = np.vstack(X_list)

print(X_all.shape)

(50000, 768)


In [40]:
np.save("imdb_bert_embeddings.npy", X_all)
np.save("imdb_labels.npy", y)

In [43]:
X = np.load("imdb_bert_embeddings.npy")
y = np.load("imdb_labels.npy")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=44,
    stratify=y
)

model_lr = LogisticRegression(max_iter=2000)
model_lr.fit(X_train, y_train)

y_pred = model_lr.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8167
              precision    recall  f1-score   support

           0       0.81      0.82      0.82      5000
           1       0.82      0.81      0.82      5000

    accuracy                           0.82     10000
   macro avg       0.82      0.82      0.82     10000
weighted avg       0.82      0.82      0.82     10000



In [46]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

model_svm = SVC(kernel="linear")

model_svm.fit(X_train, y_train)

y_pred_svm = model_svm.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

Accuracy: 0.8144
              precision    recall  f1-score   support

           0       0.81      0.82      0.82      5000
           1       0.82      0.81      0.81      5000

    accuracy                           0.81     10000
   macro avg       0.81      0.81      0.81     10000
weighted avg       0.81      0.81      0.81     10000



In [49]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

model_rf = RandomForestClassifier(
    n_estimators=500,
    random_state=44,
    n_jobs=-1
)

model_rf.fit(X_train, y_train)

y_pred_rf = model_rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Accuracy: 0.7628
              precision    recall  f1-score   support

           0       0.76      0.76      0.76      5000
           1       0.76      0.77      0.76      5000

    accuracy                           0.76     10000
   macro avg       0.76      0.76      0.76     10000
weighted avg       0.76      0.76      0.76     10000

